# Simuleringslab: fra molekyler til trykk og temperatur

## Del 2: Eksperimentene

:::{admonition} Læringsmål
:class: note

Etter denne delen skal du kunne

* måle fartsfordelingen i en gass og sammenlikne den med Maxwell og Boltzmann,
* bestemme temperaturen ut fra bevegelsen alene,
* utlede trykket fra veggkollisjoner og vise at $pV = Nk_\mathrm{B}T$ følger,
* måle den indre energien og varmekapasiteten, og forklare hvor en reell gass
  avviker fra en ideell,
* vurdere måleusikkerhet i en simulering på samme måte som i et laboratorium.
:::

Hvert eksperiment følger den samme gangen:

**A. Forutsi.** Skriv ned hva du tror, før du kjører noe.
**B. Mål.** Hent ut rådata og regn selv.
**C. Sammenlikn.** Hold målingen opp mot det analytiske uttrykket.
**D. Skaler.** Endre en variabel og se om sammenhengen holder.
**E. Konkluder.** Én setning om hva målingen faktisk viste.

Noter svarene underveis. Du trenger dem i rapporten.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bridgechem as bc

from bridgechem.constants import K_B, N_A, gas_properties

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

M = gas_properties("argon")["mass_kg"]     # massen til ett argonatom, kg
BLA, ORANSJE, GRONN = "#2b6cb0", "#c05621", "#2f855a"


def fortynnet_boks(N=150, side=18.4, T=300.0, seed=0, **kwargs):
    """En fortynnet argonboks: liten radius, så gassen er nær ideell."""
    return bc.box(N=N, size=(side,) * 3, temperature=T,
                  radius=0.08, display_scale=6, seed=seed, **kwargs)

---

# Eksperiment 1: Fartsfordelingen

Vi starter med å gi *alle* partiklene nøyaktig samme fart. Det er en tilstand
som aldri forekommer i naturen. Så lar vi dem kollidere og ser hva som skjer.

:::{admonition} A. Forutsi
:class: tip

Alle 200 partiklene starter med farten $v_\mathrm{rms}$ ved 300 K, bare i
tilfeldige retninger.

1. Vil fordelingen holde seg smal, eller bre seg ut?
2. Hvis den brer seg ut: er det like sannsynlig å finne en partikkel som er
   dobbelt så rask som gjennomsnittet, som en som er halvparten så rask?
3. Hvor lang tid tror du det tar?

Skriv ned svarene før du kjører neste celle.
:::

In [ ]:
system = bc.box(N=200, size=(12, 12, 12), temperature=300, radius=0.35,
                velocity_init="uniform_speed", seed=0)

print(f"Ved start: alle farter er {bc.analysis.speeds(system.vel)[0]:.1f} m/s")
print(f"Spredning ved start:      {np.std(bc.analysis.speeds(system.vel)):.2e} m/s")

sim = system.run(t=600, animate=False)

In [ ]:
# B. Mål: fartsfordelingen ved fire tidspunkt
farter = sim.calculate("speeds")
indekser = [0, len(farter) // 20, len(farter) // 5, -1]

# faste bokser, ellers klarer ikke histogrammet det første bildet: der har
# alle partiklene nøyaktig samme fart, så dataene har null bredde
bokser = np.linspace(0, 1100, 26)

fig, akser = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, i in zip(akser, indekser):
    ax.hist(farter[i], bins=bokser, density=True, color=BLA, alpha=0.7)
    ax.set_title(f"t = {sim.times[i]*1e12:.0f} ps")
    ax.set_xlabel("fart (m/s)")
    ax.set_xlim(0, 1100)
akser[0].set_ylabel("sannsynlighetstetthet")
plt.tight_layout();

Fordelingen bygger seg opp av seg selv. Ingen har fortalt partiklene hvordan
de skal fordele farten. Det eneste som har skjedd, er at de har kollidert.

Nå sammenlikner vi den ferdige fordelingen med det analytiske uttrykket.

:::{admonition} Maxwell og Boltzmanns fartsfordeling
:class: important

I tre dimensjoner er sannsynlighetstettheten for farten

$$f(v) = 4\pi \left(\frac{m}{2\pi k_\mathrm{B}T}\right)^{3/2}
         v^2 \exp\!\left(-\frac{mv^2}{2k_\mathrm{B}T}\right)$$

Legg merke til de to konkurrerende faktorene. Eksponentialleddet straffer høye
farter, mens $v^2$ belønner dem, fordi det er flere måter å ha en høy fart på
enn en lav. Toppen er kompromisset mellom dem.
:::

In [ ]:
# C. Sammenlikn med teorien. Vi bruker de siste 60 % av bildene, etter at
# fordelingen har roet seg, og slår dem sammen for bedre statistikk.
start = int(0.4 * len(farter))
utvalg = farter[start:].ravel()

T_malt = float(np.mean(sim.calculate("temperature")[start:]))

v = np.linspace(0, utvalg.max() * 1.05, 400)
plt.figure(figsize=(7, 4))
plt.hist(utvalg, bins=45, density=True, color=BLA, alpha=0.65, label="simulering")
plt.plot(v, bc.maxwell_boltzmann_speed(v, T_malt, M, dim=3), color=ORANSJE,
         lw=2.5, label=f"Maxwell-Boltzmann, T = {T_malt:.0f} K")
plt.xlabel("fart (m/s)"); plt.ylabel("sannsynlighetstetthet"); plt.legend();

### Tre karakteristiske farter, tre måter å regne dem ut på

Fordelingen har tre naturlige "typiske" farter, og de er ikke like.

$$v_\mathrm{p} = \sqrt{\frac{2k_\mathrm{B}T}{m}}, \qquad
  \langle v \rangle = \sqrt{\frac{8k_\mathrm{B}T}{\pi m}}, \qquad
  v_\mathrm{rms} = \sqrt{\frac{3k_\mathrm{B}T}{m}}$$

Vi skal finne dem på tre uavhengige måter: direkte fra simuleringsdataene,
ved numerisk integrasjon av $f(v)$, og fra de lukkede uttrykkene over. Får vi
tre like svar, henger teori, numerikk og simulering sammen.

In [ ]:
# (i) direkte fra dataene
v_mean_sim = utvalg.mean()
v_rms_sim = np.sqrt(np.mean(utvalg ** 2))
hist, kanter = np.histogram(utvalg, bins=60, density=True)
v_p_sim = 0.5 * (kanter[np.argmax(hist)] + kanter[np.argmax(hist) + 1])

# (ii) numerisk integrasjon av f(v) med trapesmetoden
vv = np.linspace(0, 3000, 20000)
f = bc.maxwell_boltzmann_speed(vv, T_malt, M, dim=3)
norm = np.trapezoid(f, vv)                        # skal bli 1
v_mean_num = np.trapezoid(vv * f, vv)
v_rms_num = np.sqrt(np.trapezoid(vv ** 2 * f, vv))
v_p_num = vv[np.argmax(f)]

# (iii) de analytiske uttrykkene
v_mean_teori = bc.mean_speed(T_malt, M, dim=3)
v_rms_teori = bc.rms_speed(T_malt, M, dim=3)
v_p_teori = np.sqrt(2 * K_B * T_malt / M)

print(f"Normering av f(v) (skal være 1): {norm:.6f}\n")
print(f"{'':14}{'simulering':>13}{'numerisk':>13}{'analytisk':>13}")
for navn, a, b, c in (("v_p", v_p_sim, v_p_num, v_p_teori),
                      ("<v>", v_mean_sim, v_mean_num, v_mean_teori),
                      ("v_rms", v_rms_sim, v_rms_num, v_rms_teori)):
    print(f"{navn:<14}{a:>13.1f}{b:>13.1f}{c:>13.1f}")

:::{admonition} D. Skaler: to gasser i samme boks
:class: tip

Bruk `system.set_mass(gas="helium", indices=...)` til å gjøre halvparten av
partiklene til helium, og la blandingen komme til likevekt.

1. Får de to gassene samme *fart* eller samme *kinetiske energi*?
2. Plott de to fartsfordelingene i samme figur, sammen med de teoretiske
   kurvene for hver masse ved den felles temperaturen.
3. Hva er forholdet mellom rms-fartene deres, uttrykt ved massene?

Dette er mekanismen bak Grahams effusjonslov.
:::

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

```python
s = bc.box(N=200, size=(12, 12, 12), temperature=300, radius=0.35, seed=1)
lette = np.arange(s.N) % 2 == 0
s.set_mass(gas="helium", indices=lette)
r = s.run(t=800, animate=False)

v_slutt = r.calculate("speeds")[-1]
m_he = bc.constants.gas_properties("helium")["mass_kg"]

for maske, m, navn, farge in ((lette, m_he, "helium", "#2b6cb0"),
                              (~lette, M, "argon", "#c05621")):
    E = 0.5 * m * np.mean(v_slutt[maske] ** 2)
    print(f"{navn:<7} v_rms = {np.sqrt(np.mean(v_slutt[maske]**2)):6.1f} m/s"
          f"   <E_kin> = {E/K_B:6.1f} k_B  ->  T = {2*E/(3*K_B):6.1f} K")
```

De får samme *energi*, ikke samme fart. Det er ekvipartisjon: hver frihetsgrad
får $\tfrac{1}{2}k_\mathrm{B}T$ uansett hvilken masse som sitter på den.
Derfor er

$$\frac{v_\mathrm{rms,He}}{v_\mathrm{rms,Ar}} = \sqrt{\frac{m_\mathrm{Ar}}{m_\mathrm{He}}}
  \approx \sqrt{\frac{39{,}95}{4{,}00}} \approx 3{,}16$$

Helium beveger seg drøyt tre ganger så fort. Det er hele grunnen til at lette
gasser effunderer raskere, og til at helium lekker ut av ballongen mens
argonet blir igjen.
:::

:::{admonition} E. Konkluder
:class: note

Skriv én setning: hva viste dette eksperimentet om hvor fartsfordelingen
kommer fra?
:::

---

# Eksperiment 2: Temperatur

Temperatur er den første virkelig makroskopiske størrelsen vi møter. I
simuleringen finnes den ikke som en variabel. Det finnes bare hastigheter.

:::{admonition} A. Forutsi
:class: tip

Vi setter en boks til 300 K og måler $\langle mv^2 \rangle$ over kjøringen.

1. Hvilket uttrykk vil du bruke for å komme fra $\langle mv^2\rangle$ til en
   temperatur i kelvin?
2. Hvis du i stedet hadde brukt 2D-uttrykket på 3D-dataene, ville du fått for
   høy eller for lav temperatur? Med hvilken faktor?
:::

In [ ]:
# B. Mål: temperatur fra bevegelsen alene
sim = fortynnet_boks(T=300).run(t=1000, animate=False)

v = sim.calculate("velocities")          # (n_frames, N, 3)
masse = M

# regn ut for hånd, uten å bruke calculate("temperature")
m_v2 = masse * np.sum(v ** 2, axis=-1)   # (n_frames, N)
T_for_hand = np.mean(m_v2) / (3 * K_B)

print(f"Temperatur regnet for hånd:  {T_for_hand:.2f} K")
print(f"Fra biblioteket:             {np.mean(sim.calculate('temperature')):.2f} K")
print(f"Hvis vi hadde brukt 2D:      {np.mean(m_v2)/(2*K_B):.2f} K")

Legg merke til det siste tallet. Bruker du feil antall frihetsgrader, får du
et svar som ser helt rimelig ut. Ingen feilmelding, ingen absurd verdi, bare
en temperatur som er 50 prosent for høy. Det er den typen feil som er verdt å
være redd for.

In [ ]:
# D. Skaler: er sammenhengen lineær over et bredt område?
temperaturer = np.array([100, 200, 300, 500, 800, 1200], dtype=float)
E_kin_per_partikkel = []

for T in temperaturer:
    r = fortynnet_boks(T=T).run(t=400, animate=False)
    E_kin_per_partikkel.append(np.mean(r.calculate("kinetic_energy")) / r.n_particles)

E_kin_per_partikkel = np.array(E_kin_per_partikkel)

# lineær regresjon: stigningstallet skal bli (3/2) k_B
stigning, skjaering = np.polyfit(temperaturer, E_kin_per_partikkel, 1)

plt.figure(figsize=(6.5, 4))
plt.plot(temperaturer, E_kin_per_partikkel / K_B, "o", color=BLA, ms=8,
         label="simulering")
plt.plot(temperaturer, (stigning * temperaturer + skjaering) / K_B, "-",
         color=ORANSJE, lw=2,
         label=f"tilpasning: {stigning/K_B:.4f} $k_B$ per K")
plt.xlabel("temperatur (K)")
plt.ylabel(r"$\langle E_{kin}\rangle$ per partikkel ($k_B$ K)")
plt.legend()

print(f"Stigningstall: {stigning/K_B:.4f} k_B per K   (teori: 1.5)")
print(f"Skjæringspunkt: {skjaering/K_B:.4f} k_B K     (teori: 0)")

:::{admonition} Underveisoppgave: den absolutte nullpunktet
:class: tip

Skjæringspunktet med $x$-aksen i denne grafen er en måling av det absolutte
nullpunktet, i den forstand at det er temperaturen der all
translasjonsbevegelse opphører.

1. Hva blir skjæringspunktet i din tilpasning?
2. Forklar hvorfor det måtte bli der, gitt hvordan simuleringen definerer
   temperatur. Er dette en *måling* av nullpunktet, eller en konsekvens av
   definisjonen? Diskuter kort.
:::

:::{admonition} Løsningsforslag
:class: dropdown

Skjæringspunktet ligger i praksis på null, og det *måtte* det gjøre. I
simuleringen er temperaturen definert som $T = \langle mv^2\rangle/(3k_B)$, så
$T = 0$ og $\langle E_\mathrm{kin}\rangle = 0$ er samme utsagn. Vi har ikke
målt noe uavhengig.

Det interessante er heller det motsatte veien. Historisk gikk man motsatt vei:
man målte $pV$ mot temperatur for reelle gasser, ekstrapolerte den rette
linja, og fant at den traff null ved samme temperatur uansett hvilken gass man
brukte. *Det* var en måling, og den pekte mot at temperatur måtte være noe
mekanisk. Vi kommer nærmere den varianten i eksperiment 3.
:::

:::{admonition} E. Konkluder
:class: note

Én setning om forholdet mellom temperatur og bevegelse i denne modellen.
:::

---

# Eksperiment 3: Trykk fra kollisjoner

Dette er kjernen i labben. Vi skal ikke be om trykket. Vi skal telle støt.

:::{admonition} A. Forutsi
:class: tip

Én partikkel med hastighet $v_x$ treffer en vegg vinkelrett på $x$ og spretter
elastisk tilbake.

1. Hvor mye bevegelsesmengde overfører den til veggen?
2. Hvis boksen har sidelengde $L$ og partikkelen ikke kolliderer med noe annet,
   hvor ofte treffer den den samme veggen?
3. Sett dette sammen til et uttrykk for kraften fra én partikkel, og deretter
   for trykket fra $N$ partikler. Hva må du anta om $\langle v_x^2\rangle$
   kontra $\langle v^2 \rangle$?

Denne utledningen skal inn i rapporten. Gjør den nå, på papir.
:::

In [ ]:
# B. Mål: summer bevegelsesmengden partiklene gir veggene
sim = fortynnet_boks(N=150, side=18.4, T=300).run(t=2000, animate=False)

impuls = sim.wall_collisions()             # (3,) kg m/s, per akse
veggareal = sim.volume / sim.L             # (3,) m^2 per vegg

P_akse = impuls / (sim.total_time * 2 * veggareal)
P_malt = float(np.mean(P_akse))

print("Trykk målt akse for akse (Pa):", np.round(P_akse, 1))
print(f"Gjennomsnitt over aksene:      {P_malt:.1f} Pa = {P_malt/1e5:.4f} bar")
print(f"\nIdeell gasslov N k_B T / V:    {sim.ideal_gas_pressure():.1f} Pa")
print(f"Virialmetoden:                 {sim.pressure('virial'):.1f} Pa")

At de tre aksene gir nesten samme tall er i seg selv et resultat. Gassen er
isotrop: den vet ikke hvilken vei som er opp.

:::{admonition} Underveisoppgave: hvorfor spriker vegg og virial litt?
:class: tip

Veggmetoden og virialmetoden er to uavhengige veier til det samme trykket, men
de gir sjelden helt like tall.

1. Kjør den samme boksen med `radius = 0.08, 0.3, 0.6` nm og regn ut
   forholdet mellom de to metodene hver gang.
2. Hvilken vei går avviket, og hvordan avhenger det av radien?
3. Foreslå en forklaring. Hint: hvor nær veggen kan *sentrum* i en partikkel
   med radius $r$ komme?
:::

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

```python
for radius in (0.08, 0.3, 0.6):
    s = bc.box(N=150, size=(18.4,)*3, temperature=300, radius=radius, seed=0)
    r = s.run(t=1500, animate=False)
    print(f"r = {radius:.2f} nm:  vegg/virial = "
          f"{r.pressure('wall')/r.pressure('virial'):.4f}")
```

Veggmetoden leser høyest, og avviket vokser med radien.

Forklaringen ligger i volumet. Sentrum i en partikkel med radius $r$ kan ikke
komme nærmere veggen enn $r$, så volumet som faktisk er tilgjengelig for
sentrene er $(L-2r)^3$, ikke $L^3$. Veggmetoden måler den kraften som faktisk
virker på veggen, mens virialformelen deler på hele $L^3$. Ved liten radius
forsvinner forskjellen.

Dette er ikke en feil i simuleringen. Det er den samme fysikken som gir
$b$-leddet i van der Waals, sett fra en litt annen kant.
:::

In [ ]:
# D. Skaler I: varier N ved fast volum og temperatur
N_verdier = np.array([40, 80, 120, 160, 200])
P_av_N = []

for N in N_verdier:
    r = fortynnet_boks(N=int(N), side=18.4, T=300).run(t=1500, animate=False)
    P_av_N.append(r.pressure("wall"))

P_av_N = np.array(P_av_N)
stigning_N, _ = np.polyfit(N_verdier, P_av_N, 1)

plt.figure(figsize=(6.5, 4))
plt.plot(N_verdier, P_av_N / 1e5, "o", color=BLA, ms=8, label="målt")
plt.plot(N_verdier, np.polyval(np.polyfit(N_verdier, P_av_N, 1), N_verdier) / 1e5,
         "-", color=ORANSJE, lw=2, label="lineær tilpasning")
plt.xlabel("antall partikler N"); plt.ylabel("trykk (bar)"); plt.legend()

V = (18.4e-9) ** 3
print(f"Målt stigningstall dP/dN:  {stigning_N:.4e} Pa")
print(f"Teori k_B T / V:           {K_B*300/V:.4e} Pa")
print(f"Forhold:                   {stigning_N/(K_B*300/V):.4f}")

In [ ]:
# D. Skaler II: varier volumet ved fast N og T. Boyles lov.
sider = np.array([14.0, 16.0, 18.4, 21.0, 24.0])
P_av_V, volumer = [], []

for side in sider:
    r = fortynnet_boks(N=150, side=float(side), T=300).run(t=1500, animate=False)
    P_av_V.append(r.pressure("wall"))
    volumer.append(r.volume)

P_av_V, volumer = np.array(P_av_V), np.array(volumer)
stigning_V, skjaering_V = np.polyfit(1 / volumer, P_av_V, 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(volumer * 1e27, P_av_V / 1e5, "o-", color=BLA, ms=7)
ax1.set_xlabel("volum (nm$^3$)"); ax1.set_ylabel("trykk (bar)")
ax1.set_title("P mot V")
ax2.plot(1 / volumer, P_av_V / 1e5, "o", color=BLA, ms=8)
ax2.plot(1 / volumer, (stigning_V / volumer + skjaering_V) / 1e5, "-",
         color=ORANSJE, lw=2)
ax2.set_xlabel("1/V (m$^{-3}$)"); ax2.set_ylabel("trykk (bar)")
ax2.set_title("P mot 1/V blir en rett linje")
plt.tight_layout()

print(f"Målt stigningstall d(P)/d(1/V): {stigning_V:.4e} Pa m^3")
print(f"Teori N k_B T:                  {150*K_B*300:.4e} Pa m^3")
print(f"Forhold:                        {stigning_V/(150*K_B*300):.4f}")

:::{admonition} D. Skaler III: der den ideelle gassloven svikter
:class: tip

Nå gjør vi det motsatte av å være forsiktige. Øk `radius` gradvis og se hvor
langt trykket vandrer fra $Nk_\mathrm{B}T/V$.

1. Kjør med `radius = 0.08, 0.2, 0.4, 0.6, 0.8` nm ved fast $N$, $V$ og $T$.
   Plott kompressibilitetsfaktoren $Z = PV/(Nk_\mathrm{B}T)$ mot
   pakningsgraden $\eta = N \cdot \tfrac{4}{3}\pi r^3 / V$.
2. For harde kuler er den ledende korreksjonen $Z \approx 1 + 4\eta$. Legg den
   inn i figuren. Hvor langt holder den?
3. Van der Waals skriver $p(V - nb) = nRT$ for en gass uten tiltrekning.
   Bruk målingene dine til å anslå $b$ i enheten m$^3$/mol, og sammenlikn med
   tabellverdien for argon, som er omtrent $3{,}2 \cdot 10^{-5}$ m$^3$/mol.
:::

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

```python
radier = np.array([0.08, 0.2, 0.4, 0.6, 0.8])
Z, eta = [], []
for radius in radier:
    s = bc.box(N=150, size=(18.4,)*3, temperature=300, radius=float(radius), seed=0)
    r = s.run(t=1500, animate=False)
    Z.append(r.pressure("virial") / r.ideal_gas_pressure())
    eta.append(150 * (4/3) * np.pi * (radius*1e-9)**3 / r.volume)
Z, eta = np.array(Z), np.array(eta)

plt.plot(eta, Z, "o-", label="simulering")
plt.plot(eta, 1 + 4*eta, "--", label=r"$1 + 4\eta$")
plt.xlabel(r"pakningsgrad $\eta$"); plt.ylabel("Z"); plt.legend()

# b fra det ekskluderte volumet per mol
b = 4 * (4/3) * np.pi * (0.17e-9)**3 * 6.022e23
print(f"b anslått fra en hardkuleradius på 0,17 nm: {b:.2e} m^3/mol")
```

$1 + 4\eta$ følger målingene godt opp til noen få prosent pakning og
underestimerer deretter. Det er ventet: det er bare første ledd i en
virialrekke.

For $b$: det ekskluderte volumet for et par harde kuler er *fire ganger* det
egne volumet til én kule, ikke ett. To kuler med radius $r$ kan ikke komme
nærmere hverandre enn $2r$, så hver av dem stenger av en kule med radius $2r$,
og halvparten tilskrives hver partikkel. Med $r \approx 0{,}17$ nm gir det
$b \approx 5 \cdot 10^{-5}$ m$^3$/mol, samme størrelsesorden som
tabellverdien. Avviket skyldes at ekte argon også tiltrekker hverandre, noe vi
skrur på i eksperiment 4.
:::

:::{admonition} E. Konkluder
:class: note

Én setning om hvor $pV = Nk_\mathrm{B}T$ kommer fra.
:::

---

# Eksperiment 4: Indre energi og reelle gasser

Til nå har partiklene bare kollidert. De har ikke *følt* hverandre på avstand.
Nå skrur vi på Lennard-Jones-kreftene og ser hva det gjør med den indre
energien.

:::{admonition} A. Forutsi
:class: tip

1. For en enatomig ideell gass, hva er $U$ uttrykt ved $N$, $k_\mathrm{B}$ og
   $T$? Hva blir $C_V$?
2. Hva er $(\partial U/\partial V)_T$ for en ideell gass? Begrunn med
   utgangspunkt i hva $U$ består av.
3. Når vi skrur på tiltrekning mellom partiklene, blir $U$ større eller mindre
   ved samme temperatur?
:::

In [ ]:
# B. Mål: U og C_V for den ideelle gassen
temperaturer = np.array([150, 250, 350, 450, 600], dtype=float)
U_ideell = []

for T in temperaturer:
    r = fortynnet_boks(N=150, T=float(T)).run(t=500, animate=False)
    U = np.mean(r.calculate("total_energy"))     # kinetisk + potensiell
    U_ideell.append(U)

U_ideell = np.array(U_ideell)

# C_V som den numeriske deriverte dU/dT, med sentraldifferanser
C_V_num = np.gradient(U_ideell, temperaturer)
C_V_teori = 1.5 * 150 * K_B

print(f"{'T (K)':>8}{'U (J)':>14}{'C_V numerisk':>16}{'C_V/teori':>12}")
for T, U, C in zip(temperaturer, U_ideell, C_V_num):
    print(f"{T:>8.0f}{U:>14.3e}{C:>16.3e}{C/C_V_teori:>12.4f}")

print(f"\nTeori C_V = (3/2) N k_B = {C_V_teori:.3e} J/K")
print(f"Molar: {1.5*8.314:.2f} J/(mol K)")

In [ ]:
# Er U uavhengig av volumet? Joules gamle spørsmål.
print("Samme N og T, ulike volum:")
for side in (14.0, 18.4, 24.0):
    r = fortynnet_boks(N=150, side=side, T=300).run(t=500, animate=False)
    U = np.mean(r.calculate("total_energy"))
    print(f"  side = {side:5.1f} nm   V = {r.volume*1e27:7.0f} nm^3"
          f"   U = {U:.5e} J")

Volumet endres med en faktor fem, og $U$ står stille. For en ideell gass er
$(\partial U/\partial V)_T = 0$, og her ser vi hvorfor: energien sitter bare i
bevegelsen, og bevegelsen bryr seg ikke om hvor stor boksen er.

Nå bryter vi den antakelsen.

In [ ]:
# C. Med Lennard-Jones: partiklene tiltrekker hverandre på avstand
tett = bc.box(N=150, size=(6, 6, 6), temperature=300, boundary="periodic", seed=0)
tett.add_interactions("LJ")
r_lj = tett.run(t=40, animate=False)

E_kin = np.mean(r_lj.calculate("kinetic_energy"))
E_pot = np.mean(r_lj.calculate("potential_energy"))

print(f"Kinetisk energi:  {E_kin:+.4e} J")
print(f"Potensiell energi:{E_pot:+.4e} J")
print(f"Indre energi U:   {E_kin + E_pot:+.4e} J")
print(f"\nDen potensielle energien utgjør {abs(E_pot)/(E_kin):.1%} av den kinetiske,")
print("og den er negativ. Partiklene tiltrekker hverandre i gjennomsnitt.")

In [ ]:
# D. Skaler: kjøl ned en Lennard-Jones-gass og se den kondensere
tett = bc.box(N=150, size=(6, 6, 6), temperature=300, boundary="periodic", seed=0)
tett.add_interactions("LJ")
tett.set_temperature(30, rate=10)          # 10 K per ps
kjolt = tett.run(t=40, animate=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
t_ps = kjolt.times * 1e12
ax1.plot(t_ps, kjolt.calculate("temperature"), color=BLA)
ax1.set_xlabel("tid (ps)"); ax1.set_ylabel("temperatur (K)")
ax2.plot(t_ps, kjolt.calculate("potential_energy") / 150 / K_B, color=GRONN)
ax2.set_xlabel("tid (ps)")
ax2.set_ylabel(r"$E_{pot}$ per partikkel ($k_B$ K)")
ax2.set_title("Faller når partiklene binder seg til hverandre")
plt.tight_layout();

:::{admonition} Underveisoppgave: fra kondensering til kokepunkt
:class: tip

Den potensielle energien per partikkel faller når gassen kondenserer. Fallet
er et grovt mål på bindingsenergien i væsken.

1. Les av $\Delta E_\mathrm{pot}$ per partikkel fra figuren, i joule.
2. Regn om til kJ per mol. Sammenlikn med fordampningsentalpien til argon,
   som er omtrent 6,5 kJ/mol.
3. Hvorfor er det rimelig at simuleringen bommer noe? Nevn to grunner.
:::

:::{admonition} Løsningsforslag
:class: dropdown

```python
pe = kjolt.calculate("potential_energy") / 150
delta = (pe[0] - pe[-1])
print(f"Delta E_pot per partikkel: {delta:.3e} J = {delta*6.022e23/1000:.2f} kJ/mol")
```

Størrelsesorden treffer, men tallet blir ikke eksakt, av minst to grunner.

For det første er ikke systemet i likevekt. Nedkjølingen går på titalls
pikosekunder, altså mye raskere enn en ekte kondensering, så vi fanger en
tilstand under veis snarere enn en fullt dannet væske.

For det andre er 150 partikler i en boks på 6 nm et system der nesten alt er
overflate. En dråpe av den størrelsen har en helt annen andel
overflatepartikler enn en makroskopisk væske, og overflatepartikler har færre
naboer og dermed mindre bindingsenergi.

I tillegg er fordampningsentalpi en entalpi ved konstant trykk, ikke bare en
forskjell i potensiell energi, så $\Delta H = \Delta U + p\Delta V$.
:::

:::{admonition} E. Konkluder
:class: note

Én setning om hva som skiller den indre energien til en ideell gass fra den
til en reell.
:::

---

## Ekstraoppgave: broen til kinetikken

Andelen molekyler med kinetisk energi over en terskel $E_a$ er det som styrer
reaksjonsfarten. I Arrhenius-uttrykket $k = A e^{-E_a/RT}$ er
eksponentialleddet nettopp en Boltzmann-faktor.

Finn, ved numerisk integrasjon av $f(v)$, hvilken temperatur som gjør at
akkurat 1 prosent av argonatomene har kinetisk energi over
$5{,}0 \cdot 10^{-21}$ J. Bruk halveringsmetoden eller
`scipy.optimize.brentq`, og sjekk svaret mot simuleringen ved å telle
partikler direkte.

Sammenlikn deretter andelen ved den temperaturen og ved 50 K høyere. Hvor mye
øker den? Det tallet er grunnen til at reaksjonsfart er så følsom for
temperatur.

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

```python
from scipy.optimize import brentq

E_a = 5.0e-21     # J

def andel_over(T):
    v = np.linspace(0, 5000, 40000)
    f = bc.maxwell_boltzmann_speed(v, T, M, dim=3)
    over = 0.5 * M * v**2 > E_a
    return np.trapezoid(f[over], v[over]) / np.trapezoid(f, v)

T_1prosent = brentq(lambda T: andel_over(T) - 0.01, 50, 2000)
print(f"1 % over terskelen ved T = {T_1prosent:.1f} K")

# sjekk mot simuleringen
r = fortynnet_boks(N=200, T=T_1prosent).run(t=600, animate=False)
sp = r.calculate("speeds")[len(r.times)//2:]
print(f"Simulert andel: {np.mean(0.5*M*sp**2 > E_a):.4f}")

for dT in (0, 50):
    print(f"T = {T_1prosent+dT:6.1f} K:  andel = {andel_over(T_1prosent+dT):.4f}")
```

Andelen omtrent dobles ved 50 K oppvarming. Det er den samme følsomheten som
tommelfingerregelen om at reaksjonsfarten dobles per 10 grader ved
romtemperatur, bare med en annen aktiveringsenergi.

Poenget å ta med videre: Boltzmann-faktoren du møter i kinetikken er ikke et
nytt prinsipp. Det er halen av den fartsfordelingen du målte i eksperiment 1.
:::